# U-Net · KITTI Lane Detection
**Deep Learning Assignment — Autonomous Lane Segmentation**

---
### Steps
1. Check GPU
2. Install dependencies
3. Mount Google Drive & unzip dataset
4. Verify dataset structure
5. Upload project files
6. Train the model
7. View training history
8. Run inference on a single image
9. Save model to Google Drive
10. Start API server for frontend

---
## Step 1 — Check GPU

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'Memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU — go to Runtime > Change runtime type > T4 GPU')

---
## Step 2 — Install Dependencies

In [ ]:
!pip install -q \
    segmentation-models-pytorch \
    albumentations \
    flask \
    flask-cors \
    pyngrok \
    pyyaml

print('All dependencies installed.')

---
## Step 3 — Mount Google Drive & Unzip Dataset

Upload the `data_road_224.zip` to your Google Drive first, then update `ZIP_PATH` below.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ── UPDATE THIS PATH ──────────────────────────────────────────
ZIP_PATH = '/content/drive/MyDrive/data_road_224.zip'
DEST_DIR = '/content/data'
# ─────────────────────────────────────────────────────────────

os.makedirs(DEST_DIR, exist_ok=True)
!unzip -q "{ZIP_PATH}" -d "{DEST_DIR}"

print('Unzip complete. Contents:')
for entry in os.listdir(DEST_DIR):
    print(f'  {DEST_DIR}/{entry}')

---
## Step 4 — Verify Dataset Structure

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

DATA_ROOT = '/content/data/data_road_224'
IMG_DIR   = os.path.join(DATA_ROOT, 'training', 'image_2')
MASK_DIR  = os.path.join(DATA_ROOT, 'training', 'gt_image_2')

images = sorted([f for f in os.listdir(IMG_DIR)  if f.endswith('.png')])
masks  = sorted([f for f in os.listdir(MASK_DIR) if f.endswith('.png')])

print(f'Training images : {len(images)}')
print(f'Training masks  : {len(masks)}')
print(f'Sample image    : {images[0]}')
print(f'Sample mask     : {masks[0]}')

# Display one pair
img  = np.array(Image.open(os.path.join(IMG_DIR,  images[0])).convert('RGB'))
mask = np.array(Image.open(os.path.join(MASK_DIR, masks[0])).convert('RGB'))

# Convert KITTI pink mask to binary for display
binary = ((mask[:,:,0]>150)&(mask[:,:,1]<100)&(mask[:,:,2]>150)).astype(np.uint8)*255

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(img);    axes[0].set_title('Driving Image');  axes[0].axis('off')
axes[1].imshow(mask);   axes[1].set_title('Raw KITTI Mask'); axes[1].axis('off')
axes[2].imshow(binary, cmap='gray'); axes[2].set_title('Binary Mask'); axes[2].axis('off')
plt.tight_layout()
plt.show()

---
## Step 5 — Upload Project Files

Upload `train.py` and `predict.py` from your local machine.

In [ ]:
from google.colab import files

print('Select train.py and predict.py:')
uploaded = files.upload()

for fname in uploaded:
    print(f'Uploaded: {fname}')

---
## Step 6 — Train the Model

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────
DATA_ROOT       = '/content/data/data_road_224'
MODEL_SAVE_PATH = '/content/models/lane_best.pth'
EPOCHS          = 50
BATCH_SIZE      = 8
IMAGE_SIZE      = 224
LR              = 1e-4
# ─────────────────────────────────────────────────────────────

!python train.py \
    --data_root        "{DATA_ROOT}" \
    --model_save_path  "{MODEL_SAVE_PATH}" \
    --epochs           {EPOCHS} \
    --batch_size       {BATCH_SIZE} \
    --image_size       {IMAGE_SIZE} \
    --lr               {LR}

---
## Step 7 — View Training History

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

img = mpimg.imread('logs/training_history.png')
plt.figure(figsize=(15, 4))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

---
## Step 8 — Run Inference on a Single Image

In [ ]:
import os

DATA_ROOT   = '/content/data/data_road_224'
MODEL_PATH  = '/content/models/lane_best.pth'
OUTPUT_PATH = '/content/output/prediction.png'

# Auto-pick first image
IMG_DIR = os.path.join(DATA_ROOT, 'training', 'image_2')
sample  = sorted([f for f in os.listdir(IMG_DIR) if f.endswith('.png')])[0]
SAMPLE_IMAGE = os.path.join(IMG_DIR, sample)

print(f'Running inference on: {SAMPLE_IMAGE}')

!python predict.py \
    --image_path  "{SAMPLE_IMAGE}" \
    --model_path  "{MODEL_PATH}" \
    --output_path "{OUTPUT_PATH}"

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

result = mpimg.imread(OUTPUT_PATH)
plt.figure(figsize=(15, 5))
plt.imshow(result)
plt.axis('off')
plt.tight_layout()
plt.show()

---
## Step 9 — Save Model to Google Drive

In [ ]:
import shutil, os

DRIVE_SAVE_DIR = '/content/drive/MyDrive/unet_lane_model'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

shutil.copy('/content/models/lane_best.pth',  f'{DRIVE_SAVE_DIR}/lane_best.pth')
shutil.copy('logs/training_history.png',      f'{DRIVE_SAVE_DIR}/training_history.png')
shutil.copy('logs/config.yaml',               f'{DRIVE_SAVE_DIR}/config.yaml')

print(f'Model saved to Google Drive → {DRIVE_SAVE_DIR}')

---
## Step 10 — Start API Server for Frontend

> **Run this last.** Copy the `https://...ngrok-free.app` URL and paste into your frontend `.env` as `VITE_API_URL` only if accessing remotely. For local development you can skip this.

In [ ]:
from pyngrok import ngrok
import subprocess, threading, time

MODEL_PATH = '/content/models/lane_best.pth'
PORT       = 5000

def run_server():
    subprocess.run([
        'python', 'predict.py',
        '--serve',
        '--model_path', MODEL_PATH,
        '--port', str(PORT),
        '--no_ngrok',
    ])

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(3)

public_url = ngrok.connect(PORT)
print('=' * 50)
print(f'  API URL  : {public_url}')
print(f'  Health   : {public_url}/health')
print('=' * 50)
print('Paste this URL into frontend .env as VITE_API_URL')

---
### Tips
- **Session disconnects?** Re-run Steps 2 and 10 only — model is saved to Drive.
- **Out of memory?** Reduce `BATCH_SIZE` to `4` in Step 6.
- **Slow training?** Make sure Runtime > Change runtime type is set to **T4 GPU**.
- **KITTI masks look all black?** That's normal for non-lane pixels — binary mask is correct.